# 🧠 NS-ARC Deep Transformer Scaling Experiment

**Three model variants with Attention Residual Transformer backbones at 32, 64, 128 depth — applied to every module (Encoder, World Model, Planner).**

---
| Model | Encoder Depth | World Model Depth | Planner Sims |
|-------|--------------|------------------|--------------|
| `NSARC-32`  | 4  | 32  | 25  |
| `NSARC-64`  | 8  | 64  | 50  |
| `NSARC-128` | 12 | 128 | 100 |

Each module of each model is saved separately for per-module depth analysis.

## Cell 0 — Bootstrap (Kaggle / Colab)

In [ ]:
import subprocess, sys, os

def run(cmd): subprocess.run(cmd, shell=True, check=True)

run('pip install -q wandb umap-learn scikit-learn tqdm seaborn')

# Clone Re-ARC if not present
if not os.path.isdir('data/re-arc'):
    run('git clone --quiet https://github.com/michaelhodel/re-arc data/re-arc')
    print(f'Re-ARC tasks: {len(list(__import__("pathlib").Path("data/re-arc").glob("*.json")))} files')

print('Bootstrap done')

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict
import time, json, pathlib

# ── Detect best device ──────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'Using device: {DEVICE}')
if DEVICE == 'cuda': print(torch.cuda.get_device_name(0))
torch.manual_seed(42)
np.random.seed(42)

## Cell W1 — Weights & Biases Setup

In [ ]:
import wandb

# ── Paste your key from https://wandb.ai/authorize ───────────────────────
# On Kaggle: add it as a secret (left panel → Add-ons → Secrets → WANDB_API_KEY)
WANDB_KEY = os.environ.get('WANDB_API_KEY', '')  # or paste directly as a string

if WANDB_KEY:
    wandb.login(key=WANDB_KEY)
else:
    wandb.login()   # interactive prompt

WANDB_PROJECT = 'NS-ARC-Scaling'
WANDB_ENTITY  = None   # set to your W&B team or username, or leave None
print(f'W&B project  : {WANDB_PROJECT}')
print(f'W&B entity   : {WANDB_ENTITY or "(default)"}')

## 1. Configuration: Three Depth Profiles

In [ ]:
# ── Shared hyperparameters ───────────────────────────────────────────────
BASE_CFG = dict(
    latent_dim   = 128,
    action_dim   = 10,
    hidden_dim   = 256,
    nhead        = 8,
    in_channels  = 1,
    device       = DEVICE,
    use_attn_res = True,
    ent_coef     = 0.01,
    gamma        = 0.99,
    lam          = 0.95,
    vocab_size   = 50,
    pb_c_init    = 1.25,
    pb_c_base    = 19652,
    curiosity_scale = 0.5,
)

# ── Three model profiles: depth scales all modules proportionally ─────────
PROFILES = {
    'NSARC-32': dict(**BASE_CFG,
        num_layers = 32,        # World Model depth
        enc_depth  = 4,         # Transformer Encoder depth
        num_simulations = 25,   # MCTS planner simulations
    ),
    'NSARC-64': dict(**BASE_CFG,
        num_layers = 64,
        enc_depth  = 8,
        num_simulations = 50,
    ),
    'NSARC-128': dict(**BASE_CFG,
        num_layers = 128,
        enc_depth  = 12,
        num_simulations = 100,
    ),
}

# ── Save directory ───────────────────────────────────────────────────────
SAVE_DIR = pathlib.Path('./checkpoints')
SAVE_DIR.mkdir(exist_ok=True)
for name in PROFILES:
    (SAVE_DIR / name).mkdir(exist_ok=True)

print('Profiles defined:')
for name, cfg in PROFILES.items():
    print(f'  {name}: WM depth={cfg["num_layers"]}, Enc depth={cfg["enc_depth"]}, MCTS sims={cfg["num_simulations"]}')

## 2. Environment & Dataset Setup

In [ ]:
from data.rearc_dataset import ReARCDataset
from envs.arc_env import ARCEnvironment

# ── Re-ARC: 40k+ procedurally generated grid pairs ────────────────────
REARC_PATH = os.environ.get('REARC_DATA_PATH', 'data/re-arc')
dataset    = ReARCDataset(REARC_PATH, max_pairs_per_task=100)

# ── Environment (shared for all models) ───────────────────────────────
env_cfg = {**BASE_CFG, 'rearc_path': REARC_PATH}
env     = ARCEnvironment(env_cfg)

# Quick sanity check
obs    = env.reset()
act    = env.sample_action()
o2, r, done, info = env.step(act)
sample = dataset.sample(4)
print(f'Dataset pairs   : {len(dataset)}')
print(f'State shape     : {obs["state"].shape}')
print(f'Dataset batch   : {sample["state"].shape}')
print(f'Sample reward   : {r:.4f}, done: {done}')

## 3. Build the Three Models

In [ ]:
from modules.encoders      import CNNEncoder, TransformerEncoder
from modules.world_models  import TransformerWorldModel
from modules.policies      import PPOPolicy
from modules.planners      import MCTSPlanner
from modules.curiosity     import RNDCuriosity

class DeepTransformerEncoder(TransformerEncoder):
    """TransformerEncoder that reads enc_depth from config."""
    def __init__(self, config):
        cfg = dict(config)
        cfg['_enc_depth'] = cfg.get('enc_depth', 4)
        super().__init__(cfg)
        embed_dim = cfg.get('hidden_dim', 256)
        d = cfg['_enc_depth']
        nhead = next(h for h in [8,4,2,1] if embed_dim % h == 0)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=nhead, dim_feedforward=embed_dim*4,
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=d)
        self.to(self.device)

def build_model(name: str, cfg: dict) -> dict:
    """Instantiate and return a dict of all modules for one model."""
    modules = dict(
        encoder     = DeepTransformerEncoder(cfg),
        world_model = TransformerWorldModel(cfg),
        policy      = PPOPolicy(cfg),
        planner     = MCTSPlanner(cfg),
        curiosity   = RNDCuriosity(cfg),
    )
    total = sum(p.numel() for m in modules.values() for p in m.parameters())
    print(f'{name}: total parameters = {total:,}')
    return modules

MODELS = {}
for name, cfg in PROFILES.items():
    print(f'\nBuilding {name}...')
    MODELS[name] = build_model(name, cfg)

## 4. Metric Tracker

In [ ]:
class MetricTracker:
    """Accumulates per-step metrics with rolling average display."""
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.history: dict = defaultdict(list)

    def log(self, phase: str, step: int, metrics: dict):
        for k, v in metrics.items():
            val = v.item() if hasattr(v, 'item') else float(v)
            self.history[f'{phase}/{k}'].append((step, val))

    def get(self, key: str):
        return [v for _, v in self.history.get(key, [])]

    def steps(self, key: str):
        return [s for s, _ in self.history.get(key, [])]

    def save(self, path):
        with open(path, 'w') as f:
            json.dump({k: list(v) for k, v in self.history.items()}, f, indent=2)

TRACKERS = {name: MetricTracker(name) for name in PROFILES}
print('Trackers initialized for:', list(TRACKERS.keys()))

## 5. Training Functions

In [ ]:
from training.replay_buffer import ReplayBuffer

def make_optimizers(modules: dict, lr: float = 1e-4) -> dict:
    params = [p for m in modules.values() for p in m.parameters()]
    return {'all': torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)}

def flat_state(obs) -> torch.Tensor:
    """Convert any state to [1, 1, 30, 30] for encoder input."""
    if isinstance(obs, np.ndarray):
        t = torch.tensor(obs, dtype=torch.float32)
    else:
        t = obs.float()
    while t.dim() < 4:
        t = t.unsqueeze(0)
    h, w = t.shape[-2], t.shape[-1]
    out = torch.zeros(t.shape[0], 1, 30, 30)
    out[:, 0, :min(h,30), :min(w,30)] = t[:, 0, :min(h,30), :min(w,30)] if t.shape[1]>=1 else t[:, :min(h,30), :min(w,30)]
    return out

# ── World Model pre-training on ARC task pairs ──────────────────────────
def train_world_model_epoch(
    model_name: str, modules: dict, cfg: dict,
    tracker: MetricTracker, step: int, wb_run=None,
    n_batches: int = 20, batch_size: int = 8
):
    enc = modules['encoder']
    wm  = modules['world_model']
    opt = make_optimizers({'enc': enc, 'wm': wm})
    enc.train(); wm.train()

    for b in range(n_batches):
        batch  = dataset.sample(batch_size)
        states = batch['state'].to(cfg['device'])

        z_dict = enc({'state': states})
        z = z_dict['latent']
        a = torch.randn(batch_size, cfg['action_dim'], device=cfg['device'])

        wm_out = wm({'latent': z, 'action': a,
                     'target_latent': torch.randn_like(z),
                     'target_reward': torch.zeros(batch_size, device=cfg['device'])})
        wm_loss_dict = wm.loss(
            {'target_latent': torch.randn_like(z),
             'target_reward': torch.zeros(batch_size, device=cfg['device'])},
            wm_out
        )

        opt['all'].zero_grad()
        wm_loss_dict['loss'].backward()
        torch.nn.utils.clip_grad_norm_(list(enc.parameters()) + list(wm.parameters()), 1.0)
        opt['all'].step()

        metrics = {k: v for k, v in wm_loss_dict.items()}
        if 'attention_entropy' in wm_out:
            metrics['attn_entropy'] = wm_out['attention_entropy']
        tracker.log('world_model', step + b, metrics)
        if wb_run:
            wb_run.log({f'wm/{k}': (v.item() if hasattr(v,'item') else float(v)) for k, v in metrics.items()},
                       step=step + b)


# ── Policy pre-training ───────────────────────────────────────────────────
def train_policy_epoch(
    model_name: str, modules: dict, cfg: dict,
    tracker: MetricTracker, step: int, wb_run=None,
    n_batches: int = 20, batch_size: int = 8
):
    enc = modules['encoder']
    pol = modules['policy']
    opt = make_optimizers({'enc': enc, 'pol': pol})
    enc.eval(); pol.train()

    for b in range(n_batches):
        batch  = dataset.sample(batch_size)
        states = batch['state'].to(cfg['device'])

        with torch.no_grad():
            z = enc({'state': states})['latent']

        pol_out      = pol({'latent': z})
        entropy_loss = -pol_out['entropy']
        value_reg    = pol_out['value'].pow(2).mean()
        loss         = entropy_loss + 0.01 * value_reg

        opt['all'].zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(pol.parameters(), 1.0)
        opt['all'].step()

        m = {'loss': loss, 'entropy': pol_out['entropy'], 'value_reg': value_reg}
        tracker.log('policy', step + b, m)
        if wb_run:
            wb_run.log({f'pol/{k}': (v.item() if hasattr(v,'item') else float(v)) for k, v in m.items()},
                       step=step + b)


# ── Curiosity training ───────────────────────────────────────────────────
def train_curiosity_epoch(
    model_name: str, modules: dict, cfg: dict,
    tracker: MetricTracker, step: int, wb_run=None,
    n_batches: int = 20, batch_size: int = 8
):
    enc = modules['encoder']
    cur = modules['curiosity']
    opt = make_optimizers({'cur': cur})
    cur.train()

    for b in range(n_batches):
        batch  = dataset.sample(batch_size)
        states = batch['state'].to(cfg['device'])

        with torch.no_grad():
            z = enc({'state': states})['latent']

        cur_out  = cur({'latent': z})
        cur_loss = cur.loss({}, cur_out)

        opt['all'].zero_grad()
        cur_loss['loss'].backward()
        opt['all'].step()

        m = {'rnd_loss': cur_out['rnd_loss'],
             'intrinsic_reward': cur_out['intrinsic_reward'].mean()}
        tracker.log('curiosity', step + b, m)
        if wb_run:
            wb_run.log({f'cur/{k}': (v.item() if hasattr(v,'item') else float(v)) for k, v in m.items()},
                       step=step + b)


# ── End-to-end training ──────────────────────────────────────────────────
def train_end_to_end_epoch(
    model_name: str, modules: dict, cfg: dict,
    tracker: MetricTracker, step: int, wb_run=None,
    n_batches: int = 20, batch_size: int = 8
):
    enc = modules['encoder']
    wm  = modules['world_model']
    pol = modules['policy']
    cur = modules['curiosity']
    for m in modules.values(): m.train()

    all_params = [p for m in modules.values() for p in m.parameters()]
    opt = torch.optim.AdamW(all_params, lr=3e-5, weight_decay=1e-4)

    for b in range(n_batches):
        batch  = dataset.sample(batch_size)
        states = batch['state'].to(cfg['device'])

        z_dict  = enc({'state': states})
        z       = z_dict['latent']
        a       = torch.randn(batch_size, cfg['action_dim'], device=cfg['device'])

        wm_out = wm({'latent': z, 'action': a,
                     'target_latent': torch.randn_like(z),
                     'target_reward': torch.zeros(batch_size, device=cfg['device'])})
        wm_loss = wm.loss(
            {'target_latent': torch.randn_like(z),
             'target_reward': torch.zeros(batch_size, device=cfg['device'])},
            wm_out
        )['loss']

        pol_out  = pol({'latent': z.detach()})
        pol_loss = -pol_out['entropy']

        cur_out  = cur({'latent': z.detach()})
        cur_loss = cur.loss({}, cur_out)['loss']

        total_loss = wm_loss + 0.1 * pol_loss + 0.05 * cur_loss

        opt.zero_grad()
        total_loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(all_params, 1.0)
        opt.step()

        metrics = {
            'total_loss': total_loss, 'wm_loss': wm_loss,
            'pol_loss': pol_loss, 'cur_loss': cur_loss,
            'grad_norm': grad_norm,
        }
        if 'attention_entropy' in wm_out:
            metrics['attn_entropy'] = wm_out['attention_entropy']
        tracker.log('e2e', step + b, metrics)
        if wb_run:
            wb_run.log({f'e2e/{k}': (v.item() if hasattr(v,'item') else float(v)) for k, v in metrics.items()},
                       step=step + b)


print('Training functions defined')

## 6. Training Loop — All Three Models (with W&B logging)

In [ ]:
N_EPOCHS     = 10        # reduce for quick test; increase for real experiments
BATCHES_EACH = 10        # batches per phase per epoch
BATCH_SIZE   = 4

WB_RUNS = {}   # store run handles for artifact upload later

for model_name, modules in MODELS.items():
    cfg     = PROFILES[model_name]
    tracker = TRACKERS[model_name]

    # ── Start a W&B run per model ──────────────────────────────────────
    wb_run = wandb.init(
        project = WANDB_PROJECT,
        entity  = WANDB_ENTITY,
        name    = model_name,
        config  = {
            'model':       model_name,
            'wm_depth':    cfg['num_layers'],
            'enc_depth':   cfg['enc_depth'],
            'mcts_sims':   cfg['num_simulations'],
            'latent_dim':  cfg['latent_dim'],
            'batch_size':  BATCH_SIZE,
            'n_epochs':    N_EPOCHS,
            'use_attn_res': cfg['use_attn_res'],
        },
        reinit = True
    )
    WB_RUNS[model_name] = wb_run

    print(f"\n{'='*55}\n  Training {model_name}  |  W&B: {wb_run.url}\n{'='*55}")

    for epoch in range(N_EPOCHS):
        global_step = epoch * BATCHES_EACH
        t0 = time.time()

        # Phase 1: World Model
        train_world_model_epoch(model_name, modules, cfg, tracker, global_step, wb_run, BATCHES_EACH, BATCH_SIZE)
        # Phase 2: Policy
        train_policy_epoch     (model_name, modules, cfg, tracker, global_step, wb_run, BATCHES_EACH, BATCH_SIZE)
        # Phase 3: Curiosity
        train_curiosity_epoch  (model_name, modules, cfg, tracker, global_step, wb_run, BATCHES_EACH, BATCH_SIZE)
        # Phase 4: End-to-End
        train_end_to_end_epoch (model_name, modules, cfg, tracker, global_step, wb_run, BATCHES_EACH, BATCH_SIZE)

        wm_loss  = np.mean(tracker.get('world_model/loss')[-BATCHES_EACH:] or [0])
        e2e_loss = np.mean(tracker.get('e2e/total_loss')[-BATCHES_EACH:] or [0])
        elapsed  = time.time() - t0
        print(f'  Epoch {epoch+1:>3}/{N_EPOCHS} | WM Loss: {wm_loss:.4f} | E2E Loss: {e2e_loss:.4f} | {elapsed:.1f}s')
        wb_run.log({'epoch/wm_loss': wm_loss, 'epoch/e2e_loss': e2e_loss}, step=global_step)

    # Save tracker metrics
    tracker.save(SAVE_DIR / model_name / 'metrics.json')
    print(f'  Metrics saved')
    # Note: W&B run stays open for artifact upload in Cell 7

print('\nAll models trained!')

## 7. Save Individual Module Checkpoints + Upload to W&B

In [ ]:
def save_modules(model_name: str, modules: dict):
    """Save each module's state_dict to its own .pt file."""
    for mod_name, module in modules.items():
        path = SAVE_DIR / model_name / f'{mod_name}.pt'
        torch.save({'state_dict': module.state_dict(),
                    'config': PROFILES[model_name]}, path)
    print(f'  Saved {len(modules)} modules for {model_name}')

def load_module(model_name: str, mod_name: str, module_instance):
    """Load a saved module checkpoint."""
    path = SAVE_DIR / model_name / f'{mod_name}.pt'
    ckpt = torch.load(path, map_location=PROFILES[model_name]['device'])
    module_instance.load_state_dict(ckpt['state_dict'])
    return module_instance

for name, modules in MODELS.items():
    save_modules(name, modules)

    # ── Upload to W&B as artifacts ─────────────────────────────────────
    wb_run = WB_RUNS.get(name)
    if wb_run:
        for mod_name in modules:
            pt_path = SAVE_DIR / name / f'{mod_name}.pt'
            art = wandb.Artifact(f'{name}_{mod_name}', type='model',
                                  description=f'{name} {mod_name} checkpoint')
            art.add_file(str(pt_path))
            wb_run.log_artifact(art)
        print(f'  Uploaded artifacts for {name} → {wb_run.url}')
        wb_run.finish()

print(f'\nAll checkpoints saved under {SAVE_DIR}/')
import subprocess
result = subprocess.run(['find', str(SAVE_DIR), '-name', '*.pt'], capture_output=True, text=True)
print(result.stdout)

## 8. Visualization: Per-Metric, Per-Model Dashboard

In [ ]:
COLORS = {'NSARC-32': '#4C9BE8', 'NSARC-64': '#E85D4A', 'NSARC-128': '#5CB85C'}

def plot_metric(metric_key: str, ylabel: str, title: str, ax):
    for name, tracker in TRACKERS.items():
        ys = tracker.get(metric_key)
        xs = tracker.steps(metric_key)
        if ys:
            ax.plot(xs, ys, label=name, color=COLORS[name], linewidth=1.5, alpha=0.85)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Step')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle('NS-ARC: 32 vs 64 vs 128 Layer Depth Comparison', fontsize=14, fontweight='bold')

plot_metric('world_model/loss',       'Loss',    'World Model Loss',        axes[0, 0])
plot_metric('world_model/z_loss',     'Loss',    'WM Latent Loss (MSE)',    axes[0, 1])
plot_metric('world_model/sigreg_loss','Loss',    'SIGReg (Collapse Guard)', axes[0, 2])
plot_metric('policy/entropy',         'Nats',    'Policy Entropy',          axes[1, 0])
plot_metric('policy/loss',            'Loss',    'Policy Loss',             axes[1, 1])
plot_metric('curiosity/rnd_loss',     'Loss',    'RND Curiosity Loss',      axes[1, 2])
plot_metric('curiosity/intrinsic_reward','Reward','Intrinsic Reward',       axes[2, 0])
plot_metric('e2e/total_loss',         'Loss',    'E2E Total Loss',          axes[2, 1])
plot_metric('world_model/attn_entropy','Nats',   'Attention Entropy',       axes[2, 2])

plt.tight_layout()
fig.savefig(SAVE_DIR / 'depth_comparison_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Dashboard saved')

## 9. Per-Module Depth Effect Analysis

In [ ]:
# ── Parameter count per module across depth levels ────────────────────
module_names = list(next(iter(MODELS.values())).keys())
param_table  = {m: {} for m in module_names}

for model_name, modules in MODELS.items():
    for mod_name, mod in modules.items():
        param_table[mod_name][model_name] = sum(p.numel() for p in mod.parameters())

fig2, ax2 = plt.subplots(figsize=(12, 5))
depths  = list(PROFILES.keys())
x       = np.arange(len(module_names))
width   = 0.25

for i, name in enumerate(depths):
    counts = [param_table[m].get(name, 0) / 1e6 for m in module_names]
    ax2.bar(x + i * width, counts, width, label=name, color=list(COLORS.values())[i], alpha=0.85)

ax2.set_xticks(x + width)
ax2.set_xticklabels([m.replace('_', '\n') for m in module_names])
ax2.set_ylabel('Parameters (M)')
ax2.set_title('Per-Module Parameter Count vs Depth')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
fig2.savefig(SAVE_DIR / 'per_module_params.png', dpi=150, bbox_inches='tight')
plt.show()

# Print table
print(f'\n{"Module":<16}' + ''.join(f'{n:>12}' for n in depths))
print('-' * (16 + 12 * len(depths)))
for m in module_names:
    row = f'{m:<16}' + ''.join(f'{param_table[m].get(n,0):>12,}' for n in depths)
    print(row)

## 10. Latent Space Analysis

In [ ]:
from sklearn.decomposition import PCA

fig3, axes3 = plt.subplots(1, 3, figsize=(15, 4))
fig3.suptitle('Latent Space PCA Projection per Model Depth', fontsize=13)

for ax, (name, modules) in zip(axes3, MODELS.items()):
    enc = modules['encoder']
    enc.eval()
    latents = []
    with torch.no_grad():
        for _ in range(20):
            s  = dataset.sample(8)['state'].to(PROFILES[name]['device'])
            z  = enc({'state': s})['latent'].cpu().numpy()
            latents.append(z)
    latents = np.concatenate(latents, axis=0)  # [160, D]
    proj = PCA(n_components=2).fit_transform(latents)

    ax.scatter(proj[:, 0], proj[:, 1], alpha=0.5, s=15, c=COLORS[name])
    var_exp = PCA(n_components=2).fit(latents).explained_variance_ratio_.sum()
    ax.set_title(f'{name}\nVar explained: {var_exp*100:.1f}%')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.grid(True, alpha=0.25)

plt.tight_layout()
fig3.savefig(SAVE_DIR / 'latent_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Latent PCA saved')

## 11. End-to-End Rollout Benchmark

In [ ]:
def evaluate_model(model_name: str, modules: dict, cfg: dict, n_episodes: int = 5):
    """Run n_episodes with the encoder+policy and report mean reward."""
    enc = modules['encoder']
    pol = modules['policy']
    enc.eval(); pol.eval()
    total_rewards = []

    for ep in range(n_episodes):
        obs = env.reset()
        ep_reward = 0.0
        for _ in range(50):  # max 50 steps per episode
            s_t = flat_state(obs['state']).to(cfg['device'])
            with torch.no_grad():
                z = enc({'state': s_t})['latent']
                a = pol({'latent': z})['action']
            a_vec = np.zeros(cfg['action_dim'], dtype=np.float32)
            a_vec[int(a.item()) % cfg['action_dim']] = 1.0
            obs, r, done, _ = env.step(a_vec)
            ep_reward += r
            if done: break
        total_rewards.append(ep_reward)

    return np.mean(total_rewards), np.std(total_rewards)

print(f'\n{"Model":<14} {"Mean Reward":>13} {"Std":>8}')
print('-' * 38)
eval_results = {}
for name, modules in MODELS.items():
    mean_r, std_r = evaluate_model(name, modules, PROFILES[name])
    eval_results[name] = (mean_r, std_r)
    print(f'{name:<14} {mean_r:>13.4f} {std_r:>8.4f}')

## 12. Final Summary Dashboard

In [ ]:
# ── Summary bar chart per model ──────────────────────────────────────────
fig4, axes4 = plt.subplots(1, 2, figsize=(12, 4))

names  = list(eval_results.keys())
means  = [eval_results[n][0] for n in names]
stds   = [eval_results[n][1] for n in names]
colors = [COLORS[n] for n in names]

axes4[0].bar(names, means, yerr=stds, color=colors, alpha=0.85, capsize=6)
axes4[0].set_title('Mean Episode Reward (5 episodes)')
axes4[0].set_ylabel('Reward')
axes4[0].grid(True, alpha=0.3, axis='y')

total_params = {
    n: sum(p.numel() for m in MODELS[n].values() for p in m.parameters()) / 1e6
    for n in names
}
axes4[1].bar(names, [total_params[n] for n in names], color=colors, alpha=0.85)
axes4[1].set_title('Total Parameter Count (M)')
axes4[1].set_ylabel('Parameters (M)')
axes4[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Model Depth Scaling Summary', fontsize=13)
plt.tight_layout()
fig4.savefig(SAVE_DIR / 'summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Upload summary plots to W&B eval run ─────────────────────────────────
eval_wb = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                     name='eval_summary', reinit=True)
for name in names:
    eval_wb.log({f'eval/{name}/mean_reward': eval_results[name][0],
                 f'eval/{name}/std_reward':  eval_results[name][1],
                 f'eval/{name}/params_M':    total_params[name]})
eval_wb.log({'plots/summary_dashboard':    wandb.Image(str(SAVE_DIR / 'summary_dashboard.png')),
              'plots/depth_comparison':    wandb.Image(str(SAVE_DIR / 'depth_comparison_dashboard.png')),
              'plots/latent_pca':          wandb.Image(str(SAVE_DIR / 'latent_pca.png')),
              'plots/per_module_params':   wandb.Image(str(SAVE_DIR / 'per_module_params.png'))})
eval_wb.finish()
print(f'Eval run: {eval_wb.url}')

print('\nExperiment complete! Saved outputs:')
for f in sorted(SAVE_DIR.rglob('*')):
    if not f.is_dir():
        print(f'  {f}')